In [1]:
import sys
sys.path.append('../')

from scripts.algorithms import *

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


## Read and preprocess the data

In [2]:

############
dttz = pd.read_csv("../data/student_portuguese.csv")
dtz = dttz.drop_duplicates().reset_index(drop=True)

print("Original number of rows: ", len(dtz))
print(" ")

seln = 500
dtx = dtz.sample(n=seln, replace=False, random_state=42).reset_index(drop=True)
dt = dtx.drop_duplicates().reset_index(drop=True)


############
target_column = "pass"

X = dt.drop([target_column], axis=1).values
y = dt[target_column].values
y = np.where(y == 1, 1, -1) 


scaler = StandardScaler()
X = scaler.fit_transform(X)


Original number of rows:  649
 


In [3]:
##########
np.random.seed(42)
n = X.shape[0]
indices = np.random.permutation(n)
split = int(0.9 * n)
lhs_idx = indices[:split]
rhs_idx = indices[split:]

lhs_idx_negative = lhs_idx[y[lhs_idx] == -1]
X_LHS_random = X[lhs_idx_negative]
X_RHS_random = X[rhs_idx]
y_RHS_random = y[rhs_idx]

xxneg = y[lhs_idx]

len(lhs_idx_negative), len(lhs_idx), len(rhs_idx), np.sum(xxneg == -1), np.sum(xxneg == 1)


(224, 450, 50, 224, 226)

## Run the coverage radius algorithm

In [4]:
pos_mask   = (y_RHS_random == 1)
X_RHS_pos  = X_RHS_random[pos_mask]      
pos_labels = y_RHS_random[pos_mask]     

print(f"Positive RHS nodes: {len(X_RHS_pos)}")
 
            
results = []

for R_budget in [4, 4.5, 5, 5.5, 6, 6.5, 7, 7.5, 8, 8.5, 9, 9.5, 10]:
    
    radii, covered = radius_greedy(X_LHS_random, X_RHS_pos, R_budget)
    
    print(f"\nBudget used: {R_budget}")
    print(f"Agents covered: {covered.sum()} out of {len(X_LHS_random)}")
    print("Non-zero radii for positive targets:")
    for i, r in enumerate(radii):
        if r > 0:
            print(f"  Target {i:2d}: radius {r:.3f}")
    
    row = {
        "Budget": R_budget,
        "Agents_Covered": covered.sum(),
        "Total_Agents": len(X_LHS_random)
    }

    for i, r in enumerate(radii):
        if r > 0:
            row[f"Target_{i}_Radius"] = r
    
    results.append(row)

datasf = pd.DataFrame(results)
datasf.to_csv("./cov_results/portuguese_rapproach.csv", index=False)


Positive RHS nodes: 22

Budget used: 4
Agents covered: 2 out of 224
Non-zero radii for positive targets:
  Target 10: radius 3.137

Budget used: 4.5
Agents covered: 5 out of 224
Non-zero radii for positive targets:
  Target 10: radius 4.273

Budget used: 5
Agents covered: 10 out of 224
Non-zero radii for positive targets:
  Target 10: radius 4.889

Budget used: 5.5
Agents covered: 14 out of 224
Non-zero radii for positive targets:
  Target 10: radius 5.491

Budget used: 6
Agents covered: 21 out of 224
Non-zero radii for positive targets:
  Target 10: radius 5.974

Budget used: 6.5
Agents covered: 45 out of 224
Non-zero radii for positive targets:
  Target 10: radius 6.468

Budget used: 7
Agents covered: 83 out of 224
Non-zero radii for positive targets:
  Target 10: radius 6.988

Budget used: 7.5
Agents covered: 128 out of 224
Non-zero radii for positive targets:
  Target 10: radius 7.496

Budget used: 8
Agents covered: 157 out of 224
Non-zero radii for positive targets:
  Target 10: r